# E1 — Quantization: BPW

## Research question

How does quantization (BPW) affect performance and efficiency?

## Prior hypothesis

Higher quantization (lower BPW) reduces memory but may impact throughput and perplexity.

Some models, depending on the quantization type, may "collapse".

## Experiment configuration

| Parameter | Values           |
|-----------|-------------------|
| Models | All models  |
| Quantization (BPW) | 3, 4, 5, 6, 8 bits |
| Total | Multiple runs    |


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from monitorviz.io import load_collection
from monitorviz.transforms.collection import RunCollection
from monitorviz.viz import setup_style

setup_style("talk")
warnings.filterwarnings("ignore", category=UserWarning, module="seaborn")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ── Data loading (E1) ──────────────────────────────────────────────────────────
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
DATA_ROOT = PROJECT_ROOT / "data" / "TFG-DATA" / "E1"
assert DATA_ROOT.is_dir(), f"Not found: {DATA_ROOT.resolve()}"

coll_raw = load_collection(DATA_ROOT)
print(f"Runs loaded (total): {len(coll_raw)}")

# ── Early exclusion of invalid runs (BEFORE building the DataFrames) ──────────
# DeepSeek Q4_0 and Q8_0 produce generation collapse (ASCII loop / empty
# string). Their perplexity and throughput metrics are invalid.
# NOTE: model_info.architecture = "qwen2" for DeepSeek (base architecture),
# so the check uses model_short instead, which does contain "DeepSeek".

def _is_invalid_run(run) -> bool:
    """Returns True if the run should be excluded from the E1 analysis."""
    try:
        mi    = run.summary.model_info or {}
        quant = (mi.get("quantization") or "").upper()
        if not quant:
            quant = run.model_short.upper()
        is_deepseek  = "deepseek" in run.model_short.lower()
        is_bad_quant = quant in {"Q4_0", "Q8_0"}
        return is_deepseek and is_bad_quant
    except Exception:
        return False

invalid_runs = [r for r in coll_raw.runs if _is_invalid_run(r)]
valid_runs   = [r for r in coll_raw.runs if not _is_invalid_run(r)]

if invalid_runs:
    ids = [r.run_id for r in invalid_runs]
    print(f"Runs excluded (DeepSeek Q4_0/Q8_0, invalid data): {len(invalid_runs)}")
    print(f"  IDs: {ids}")
else:
    print("No runs excluded.")

coll = RunCollection(valid_runs)
print(f"Runs in analysis: {len(coll)}")

# ── DataFrames (valid runs only) ───────────────────────────────────────────────
summary  = coll.summary_df()
hw_full  = coll.hw_metrics_df()
pm_full  = coll.prompt_metrics_df()

# ── quantization column (extracted from model_info, key for the visual system)
if "quantization" not in summary.columns:
    summary["quantization"] = [
        (r.summary.model_info or {}).get("quantization")
        for r in coll.runs
    ]

# ── bits_per_weight (lookup if OLLAMA doesn't report it) ──────────────────────
if "bits_per_weight" not in summary.columns:
    try:
        from monitorviz.transforms.aggregations import _GGUF_BPW
        bpw_list = []
        for r in coll.runs:
            mi  = r.summary.model_info
            bpw = None
            if mi:
                bpw = mi.get("bits_per_weight") or _GGUF_BPW.get(
                    (mi.get("quantization") or "").upper()
                )
            bpw_list.append(bpw)
        summary["bits_per_weight"] = bpw_list
    except Exception as e:
        print(f"bits_per_weight not available: {e}")

# Alias for compatibility with later cells
ex = summary

# ── Visual encoding system ─────────────────────────────────────────────────────
# Colors by quantization: ColorBrewer Set1.
# Non-K quants (Q4_0, Q8_0) use red/purple to visually distinguish them
# from K-quants, whose quality per bit is higher.
QUANT_ORDER  = ["Q2_K", "Q3_K_M", "Q4_0", "Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0", "BF16"]
QUANT_COLORS = {
    "Q2_K":   "#a65628",  # brown    – max K compression
    "Q3_K_M": "#377eb8",  # blue
    "Q4_0":   "#e41a1c",  # red      – non-K
    "Q4_K_M": "#ff7f00",  # orange
    "Q5_K_M": "#4daf4a",  # green
    "Q6_K":   "#f781bf",  # pink
    "Q8_0":   "#984ea3",  # purple   – non-K
    "BF16":   "#999999",  # gray     – precision reference
}
QUANT_BPW = {
    "Q2_K": 2.6, "Q3_K_M": 3.35, "Q4_0": 4.0, "Q4_K_M": 4.5,
    "Q5_K_M": 5.5, "Q6_K": 6.6,  "Q8_0": 8.0, "BF16":  16.0,
}

# Markers by model: stable assignment, alphabetical order of the label.
_MARKER_POOL = ["o", "s", "^", "D", "v", "P", "X", "*"]
MODEL_MARKERS = {
    m: _MARKER_POOL[i % len(_MARKER_POOL)]
    for i, m in enumerate(sorted(ex["model_label"].unique()))
}

print(f"Quantizations: {sorted(ex['quantization'].dropna().unique())}")
print(f"Models → markers:")
for m, mk in MODEL_MARKERS.items():
    print(f"  {m!r:35s} → '{mk}'")

# Aliases for consistency with E0 (which uses hw_df / pm_df)
hw_df = hw_full
pm_df = pm_full


In [ ]:
# ── Replace perplexity_geomean with Wikitext-2 PPL (from notebook 12) ────────
# The self-referential perplexity is replaced with the one evaluated with
# llama.cpp on Wikitext-2, context=512, using the same deduplication as nb 12.
import json as _json, re as _re
from pathlib import Path as _Path

_PPL_DIR   = PROJECT_ROOT / "data" / "TFG-DATA" / "Perplejidad"
_QUANT_RE  = _re.compile(r"-(BF16|Q\d+[_K]*[_0M]*)(?:-GGUF)?\.gguf$", _re.IGNORECASE)

_ppl_rows = []
for _rd in sorted(_PPL_DIR.iterdir()):
    _rj = _rd / "resumen.json"
    if not _rj.is_file():
        continue
    _r = _json.loads(_rj.read_text())
    _fname = _Path(_r["model"]["path"]).name
    _m = _QUANT_RE.search(_fname)
    if _m:
        _quant_ppl = _m.group(1).upper()
        _base_ppl  = _fname[: _m.start()]
    else:
        _quant_ppl = _r["model"].get("quantization", "?")
        _base_ppl  = _fname.replace(".gguf", "")
    _ppl_rows.append(dict(
        model_base=_base_ppl,
        quant=_quant_ppl,
        context=_r["annotations"]["context"],
        ppl=_r["result"]["perplexity"],
        ppl_stderr=_r["result"]["ppl_stderr"],
    ))

_ppl_df = (
    pd.DataFrame(_ppl_rows)
      .sort_values("ppl")
      .drop_duplicates(subset=["model_base", "quant", "context", "ppl"], keep="first")
      .drop_duplicates(subset=["model_base", "quant", "context"],        keep="first")
)
_ppl_df = _ppl_df[_ppl_df["context"] == 512][["model_base", "quant", "ppl", "ppl_stderr"]].copy()
_ppl_map        = _ppl_df.set_index(["model_base", "quant"])["ppl"]
_ppl_stderr_map = _ppl_df.set_index(["model_base", "quant"])["ppl_stderr"]

# Derive model_base from ex["model_short"] by removing the "-{quantization}" suffix
_q_col_ppl = "quantization" if "quantization" in ex.columns else None
if _q_col_ppl is not None:
    def _derive_base(row):
        q = row[_q_col_ppl]
        return row["model_short"].removesuffix(f"-{q}") if isinstance(q, str) else row["model_short"]

    _bases = ex.apply(_derive_base, axis=1)
    ex["perplexity_geomean"] = [
        _ppl_map.get((b, str(q).upper()), np.nan)
        for b, q in zip(_bases, ex[_q_col_ppl])
    ]
    ex["ppl_wk_stderr"] = [
        _ppl_stderr_map.get((b, str(q).upper()), np.nan)
        for b, q in zip(_bases, ex[_q_col_ppl])
    ]
    _n = ex["perplexity_geomean"].notna().sum()
    print(f"Wikitext-2 PPL assigned: {_n}/{len(ex)} runs")
    print(_ppl_df.to_string(index=False))
else:
    print("\u26a0 'quantization' column not available; perplexity_geomean not updated.")


In [ ]:
# E1 visual system: quant_label, _ql_map, scatter helpers
# QUANT_ORDER / QUANT_COLORS / QUANT_BPW / MODEL_MARKERS defined in the Setup cell.
from matplotlib.lines import Line2D as _Line2D

_q_col   = "quantization" if "quantization" in ex.columns else "model_short"
_q_col_s = _q_col
ex["quant_label"] = ex["model_label"] + "\n(" + ex[_q_col].fillna("?") + ")"
_ql_map  = ex.set_index("run_id")["quant_label"].to_dict()
print("quant_labels detected:")
print(ex[["model_label", _q_col, "quant_label"]].drop_duplicates().to_string(index=False))


def scatter_e1(ax, data, x_col, y_col, xlabel, ylabel, title):
    for _, row in data.iterrows():
        q      = str(row.get(_q_col_s, "?"))
        m      = row["model_label"]
        color  = QUANT_COLORS.get(q, "gray")
        marker = MODEL_MARKERS.get(m, "o")
        ax.scatter(row[x_col], row[y_col], color=color, marker=marker, s=100, zorder=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)


def add_e1_legend(ax, quants, models):
    quant_handles = [
        _Line2D([0], [0], marker="o", color="w",
                markerfacecolor=QUANT_COLORS.get(q, "gray"), markersize=9, label=q)
        for q in quants
    ]
    sep = _Line2D([0], [0], color="none", label="")
    model_handles = [
        _Line2D([0], [0], marker=MODEL_MARKERS.get(m, "o"), color="gray",
                markersize=9, linestyle="None", label=m)
        for m in models
    ]
    ax.legend(
        handles=quant_handles + [sep] + model_handles,
        loc="upper left", bbox_to_anchor=(1.02, 1),
        title="Color=quant / Shape=model", fontsize=8, title_fontsize=8,
    )


print("E1 visual system ready.")

# 📊 Figures and tables for the thesis

| # | Figure/Table |
|---|---|
| Fig. 1 | Pairwise heatmap: FoM_full × quantization (with speedup vs Q4_K_M) |
| Tab. 1 | E1 master table: sub-tables per model (quant., RAM, tok/s, PPL, FoM, usable) |
| Tab. 2 | Priority selector table (best FoM / max speed / minimum footprint) |
| Fig. 2 | Quality–RAM footprint curve (PPL vs GBytes per model) |
| Fig. 3 | FoM_full vs BPW per model |
| Fig. 4 | K-quant vs non-K-quant (throughput, PPL, J/token) |


In [ ]:
# ── Figure 1 — Pairwise heatmap: FoM_full × quantization ────────────────────
# Rows = models, columns = quantizations.
# Background = FoM_full (green ↑ better). Text = FoM + speedup vs Q4_K_M.
from monitorviz.transforms.fom import compute_fom_full

try:
    _fom_e1 = compute_fom_full(ex)
    _q_col  = "quantization" if "quantization" in _fom_e1.columns else "model_short"

    # Mean per (model_label, quantization)
    _grp = _fom_e1.groupby(["model_label", _q_col], dropna=True)
    _piv_fom  = _grp["fom_full"].mean().unstack(_q_col)
    _piv_toks = (
        _fom_e1.groupby(["model_label", _q_col])["tokens_per_s_mean"]
        .mean().unstack(_q_col)
    )

    # Sort columns by BPW if available
    _bpw_map = {q: QUANT_BPW.get(q, 99) for q in _piv_fom.columns}
    _col_order = sorted(_piv_fom.columns, key=lambda q: _bpw_map.get(q, 99))
    _piv_fom  = _piv_fom.reindex(columns=_col_order)
    _piv_toks = _piv_toks.reindex(columns=_col_order)

    # Speedup relative to Q4_K_M
    _ref_q = "Q4_K_M"
    if _ref_q in _piv_toks.columns:
        _piv_speedup = _piv_toks.div(_piv_toks[_ref_q], axis=0)
    else:
        _piv_speedup = None

    _vals = _piv_fom.values.astype(float)
    _nrows, _ncols = _vals.shape
    _vmin = float(min(v for v in _vals.flatten() if not (v != v)))
    _vmax = float(max(v for v in _vals.flatten() if not (v != v)))

    fig, ax = plt.subplots(figsize=(max(8, _ncols * 1.6), max(3.5, _nrows * 0.7 + 1)))
    _cmap = plt.cm.RdYlGn
    _im = ax.imshow(_vals, cmap=_cmap, aspect="auto",
                    vmin=_vmin, vmax=_vmax)
    ax.grid(False)

    ax.minorticks_off()
    ax.set_xticks(range(_ncols))
    ax.set_xticklabels(_piv_fom.columns, fontsize=9)
    ax.set_yticks(range(_nrows))
    ax.set_yticklabels(_piv_fom.index, fontsize=9)
    ax.set_xlabel("Quantization", fontsize=10)
    fig.suptitle("E1 — FoM_full by model × quantization\n"
                 "(background: FoM_full  ·  text: ×speedup vs Q4_K_M)", fontsize=11)

    for r in range(_nrows):
        for c in range(_ncols):
            v = _vals[r, c]
            if v != v:   # NaN
                ax.text(c, r, "—", ha="center", va="center", fontsize=8, color="gray")
                continue
            _norm = (v - _vmin) / (_vmax - _vmin + 1e-9)
            _fg = "white" if _norm < 0.4 else "black"
            _sp_txt = ""
            if _piv_speedup is not None:
                _sp = _piv_speedup.iloc[r, c]
                if _sp == _sp:
                    _sp_txt = f"\n×{_sp:.2f}"
            ax.text(c, r, f"{v:.3f}{_sp_txt}", ha="center", va="center",
                    fontsize=7.5, color=_fg, fontweight="bold")

    plt.colorbar(_im, ax=ax, label="FoM_full", shrink=0.8)
    fig.tight_layout()
    plt.show()

except Exception as e:
    import traceback; traceback.print_exc()


In [ ]:
# ── Table 2 — Priority selector ───────────────────────────────────────────────
# 3 rows: best FoM / max usable speed / minimum footprint with acceptable PPL
from monitorviz.transforms.fom import compute_fom_full, compute_usability

try:
    _fom_s = compute_fom_full(ex)
    _q_col = "quantization" if "quantization" in _fom_s.columns else "model_short"

    # Usability
    _pm_s = pm_full if "pm_full" in dir() else (pm_full if "pm_full" in vars() else None)
    _pm_s = _pm_s if _pm_s is not None else (pm_df if "pm_df" in vars() else None)
    if _pm_s is not None and not _pm_s.empty:
        _fom_s, _ = compute_usability(_fom_s, _pm_s)
    else:
        _fom_s["usable"] = False

    def _fmt_row(row, crit):
        q = row.get(_q_col, "—") if isinstance(row, dict) else row[_q_col] if _q_col in row.index else "—"
        return {
            "Criterion": crit,
            "Model":     row["model_label"],
            "Quant.":    q,
            "FoM_full":  f"{row['fom_full']:.3f}"  if "fom_full"  in row.index and row["fom_full"] == row["fom_full"]  else "—",
            "tok/s":     f"{row['tokens_per_s_mean']:.1f}" if "tokens_per_s_mean" in row.index and row["tokens_per_s_mean"] == row["tokens_per_s_mean"] else "—",
            "RAM (GBytes)":  f"{row['model_size_gb']:.2f}" if "model_size_gb" in row.index and row["model_size_gb"] == row["model_size_gb"] else "—",
            "PPL":       f"{row['perplexity_geomean']:.2f}" if "perplexity_geomean" in row.index and row["perplexity_geomean"] == row["perplexity_geomean"] else "—",
            "Usable":    "✓" if row.get("usable", False) else "✗",
        }

    _rows = []

    # Row 1: best global FoM_full
    _best = _fom_s.dropna(subset=["fom_full"])
    if not _best.empty:
        _rows.append(_fmt_row(_best.loc[_best["fom_full"].idxmax()], "① Best FoM_full"))

    # Row 2: max tok/s among usable runs
    _usable = _fom_s[_fom_s.get("usable", False) == True] if "usable" in _fom_s.columns else pd.DataFrame()
    if _usable.empty:
        _usable = _fom_s  # fallback if none are usable
    _spd = _usable.dropna(subset=["tokens_per_s_mean"])
    if not _spd.empty:
        _rows.append(_fmt_row(_spd.loc[_spd["tokens_per_s_mean"].idxmax()], "② Max speed (usable)"))

    # Row 3: minimum RAM footprint
    _ram = _fom_s.dropna(subset=["model_size_gb"])
    if not _ram.empty:
        _rows.append(_fmt_row(_ram.loc[_ram["model_size_gb"].idxmin()], "③ Min. RAM footprint"))

    if _rows:
        _sel_df = pd.DataFrame(_rows).set_index("Criterion")
        print("Table 2 — Priority selector (E1)")
        display(_sel_df)
    else:
        print("No data for selector table.")

except Exception as e:
    import traceback; traceback.print_exc()


In [ ]:
# ── Table 1 — E1 characterization: sub-tables per model ─────────────────────
# One small table per model, rows = quantizations sorted by BPW ↓
try:
    # Reuse _fom_s from the Selector if it already exists, otherwise recompute
    if "_fom_s" not in vars():
        from monitorviz.transforms.fom import compute_fom_full, compute_usability
        _fom_s = compute_fom_full(ex)
        _pm_t = pm_full if "pm_full" in vars() else (pm_df if "pm_df" in vars() else None)
        if _pm_t is not None and not _pm_t.empty:
            _fom_s, _ = compute_usability(_fom_s, _pm_t)
        else:
            _fom_s["usable"] = False

    # σ from per-prompt metrics (intra-run variability)
    _pm_t1 = pm_full if "pm_full" in vars() else (pm_df if "pm_df" in vars() else pd.DataFrame())
    if not _pm_t1.empty and "run_id" in _pm_t1.columns and "run_id" in _fom_s.columns:
        _valid_pm1 = (_pm_t1[~_pm_t1["is_empty_generation"]]
                      if "is_empty_generation" in _pm_t1.columns else _pm_t1)
        _std_cols1 = [c for c in ["tokens_per_second", "time_to_first_token_ms", "perplexity"]
                      if c in _valid_pm1.columns]
        if _std_cols1:
            _std_df1 = (
                _valid_pm1.groupby("run_id")[_std_cols1].std().reset_index()
                .rename(columns={"tokens_per_second":      "tokens_per_s_std",
                                 "time_to_first_token_ms": "ttft_ms_std",
                                 "perplexity":             "perplexity_std"})
            )
            _fom_s = _fom_s.merge(_std_df1, on="run_id", how="left")

    # Runtime RAM (mean hw_metrics) in metric GB
    _hw_e1 = hw_full if "hw_full" in vars() else pd.DataFrame()
    if not _hw_e1.empty and "mem_used_bytes" in _hw_e1.columns and "run_id" in _fom_s.columns:
        _ram_e1 = _hw_e1.groupby("run_id")["mem_used_bytes"].mean().reset_index()
        _ram_e1["ram_used_gb"] = _ram_e1["mem_used_bytes"] / 1e9
        _fom_s = _fom_s.merge(_ram_e1[["run_id", "ram_used_gb"]], on="run_id", how="left")

    # Maximum swap used during the run (in metric GB)
    if not _hw_e1.empty and "swap_used_bytes" in _hw_e1.columns and "run_id" in _fom_s.columns:
        _swap_e1 = _hw_e1.groupby("run_id")["swap_used_bytes"].max().reset_index()
        _swap_e1["swap_max_gb"] = _swap_e1["swap_used_bytes"] / 1e9
        _fom_s = _fom_s.merge(_swap_e1[["run_id", "swap_max_gb"]], on="run_id", how="left")

    _q_col_t = "quantization" if "quantization" in _fom_s.columns else "model_short"
    _has_bpw = "bits_per_weight" in _fom_s.columns

    _COL_MAP = {
        _q_col_t:             "Quant.",
        "bits_per_weight":    "BPW",
        "ram_used_gb":        "RAM (GBytes)",
        "swap_max_gb":        "Max swap (GBytes)",
        "tokens_per_s_mean":  "tok/s",
        "tokens_per_s_std":   "σ tok/s",
        "ttft_ms_mean":       "TTFT (ms)",
        "ttft_ms_std":        "σ TTFT",
        "energy_per_token_j": "J/tok",
        "mbu_corr":           "MBU (%)",
        "cpu_efficiency":     "η_CPU",
        "perplexity_geomean": "PPL",
        "ppl_wk_stderr":      "σ PPL_wk",
        #"perplexity_std":     "σ PPL",
        "fom_full":           "FoM_full",
        "usable":             "Usable",
    }
    _src_cols = [c for c in _COL_MAP if c in _fom_s.columns]

    _ROUND = {
        "BPW": 1, "RAM (GBytes)": 2, "Max swap (GBytes)": 2,
        "tok/s": 1, "σ tok/s": 1,
        "TTFT (ms)": 0, "σ TTFT": 0,
        "J/tok": 3, "MBU (%)": 1, "η_CPU": 3,
        "PPL": 2, "σ PPL_wk": 2, "σ PPL": 2, "FoM_full": 3,
    }

    for _mdl in sorted(_fom_s["model_label"].unique()):
        _sub = _fom_s[_fom_s["model_label"] == _mdl][_src_cols].copy()
        _sub = _sub.rename(columns=_COL_MAP)
        if _has_bpw and "BPW" in _sub.columns:
            _sub = _sub.sort_values("BPW", ascending=False)
        if "Usable" in _sub.columns:
            _sub["Usable"] = _sub["Usable"].map({True: "✓", False: "✗"})
        for col, dec in _ROUND.items():
            if col in _sub.columns:
                _sub[col] = _sub[col].round(dec)
        if "Quant." in _sub.columns:
            _sub = _sub.set_index("Quant.")
        print(f"\n── {_mdl}")
        display(_sub)

except Exception:
    import traceback; traceback.print_exc()

In [ ]:
# Quality–RAM footprint frontier: ram_used_gb (runtime) vs perplexity_geomean
_ex_ram_q = ex.copy()
_hw_q = hw_full if "hw_full" in vars() else pd.DataFrame()
if not _hw_q.empty and "mem_used_bytes" in _hw_q.columns and "run_id" in _ex_ram_q.columns:
    _rq = _hw_q.groupby("run_id")["mem_used_bytes"].mean().reset_index()
    _rq["ram_used_gb"] = _rq["mem_used_bytes"] / 1e9
    _ex_ram_q = _ex_ram_q.merge(_rq[["run_id","ram_used_gb"]], on="run_id", how="left")

if "ram_used_gb" in _ex_ram_q.columns and "perplexity_geomean" in _ex_ram_q.columns:
    _mem_q = _ex_ram_q.dropna(subset=["ram_used_gb", "perplexity_geomean"]).copy()
    if not _mem_q.empty:
        fig, ax = plt.subplots(figsize=(11, 7))

        scatter_e1(ax, _mem_q, "ram_used_gb", "perplexity_geomean",
                   "RAM at runtime (GBytes)", "Perplexity (Wikitext-2, ctx=512)",
                   "Quality–RAM footprint frontier")

        # Barras de error verticales con ppl_wk_stderr
        if "ppl_wk_stderr" in _mem_q.columns:
            _err_q = _mem_q.dropna(subset=["ppl_wk_stderr"])
            ax.errorbar(
                _err_q["ram_used_gb"], _err_q["perplexity_geomean"],
                yerr=_err_q["ppl_wk_stderr"],
                fmt="none", color="gray", capsize=4, linewidth=1, alpha=0.6,
                label="±stderr (chunks)", zorder=2,
            )

        # Tendencia lineal global
        _z = np.polyfit(_mem_q["ram_used_gb"], _mem_q["perplexity_geomean"], 1)
        _xr = np.linspace(_mem_q["ram_used_gb"].min(), _mem_q["ram_used_gb"].max(), 100)
        ax.plot(_xr, np.polyval(_z, _xr), ls="--", color="gray", alpha=0.5,
                label="Linear trend", zorder=1)

        # Reference line: total system RAM
        ax.axvline(8.0, ls=":", color="red", alpha=0.6, label="Total RAM (8 GBytes)")

        _quants_mem = [q for q in QUANT_ORDER if q in _mem_q[_q_col_s].values]
        _models_mem = sorted(_mem_q["model_label"].unique())
        add_e1_legend(ax, _quants_mem, _models_mem)

        fig.suptitle("E1 — Quality–RAM footprint curve (\u2199 = lower PPL and lower GBytes = better)",
                     fontsize=12, y=1.02)
        fig.tight_layout()
        plt.show()
    else:
        print("No data for quality–footprint frontier")
else:
    print("Columns ram_used_gb / perplexity_geomean not available")


## Figure of Merit (FoM_full)

Combines four normalized axes relative to the reference configuration
(**llama3.2:3b · llama.cpp · Q4_K_M · ctx=4096 · bs=512**):

$$\text{FoM\_full} = \bigl(T_{norm} \cdot MBU_{norm} \cdot \eta_{norm} \cdot \varepsilon_{norm}\bigr)^{1/4}$$

| Axis | Normalization | Meaning |
|-----|--------------|-------------|
| $T_{norm}$ | $N \cdot T / (N_{ref} T_{ref})$ | Size-weighted throughput |
| $MBU_{norm}$ | $\text{MBU\_corr} / \text{MBU}_{ref}$ | Bandwidth utilization (corrected) |
| $\eta_{norm}$ | $\eta_{CPU} / \eta_{ref}$ | Computational efficiency |
| $\varepsilon_{norm}$ | $N \cdot \varepsilon / (N_{ref} \varepsilon_{ref})$ | Size-weighted tokens/J |

**FoM_full > 1** → the configuration exceeds the reference in geometric mean.


In [ ]:
# ── Figure 4 — FoM_full vs BPW per model ─────────────────────────────────────
# x = quantization (categorical axis ordered by median BPW), y = FoM_full.
# ★ = usable configuration (≥ T_hum).
from monitorviz.transforms.fom import compute_fom_full, compute_usability

try:
    fom_df = compute_fom_full(ex)

    _pm_fom = (pm_full if "pm_full" in dir() and not pm_full.empty else
               pm_df   if "pm_df"   in dir() and not pm_df.empty   else pd.DataFrame())
    if not _pm_fom.empty:
        fom_df, _tpw = compute_usability(fom_df, _pm_fom)
    else:
        fom_df["usable"] = False

    _qc = "quantization" if "quantization" in fom_df.columns else "model_short"
    _need = [_qc, "bits_per_weight", "fom_full", "model_label"]
    _fd = fom_df.dropna(subset=_need)

    if _fd.empty:
        print("No BPW/FoM data.")
    else:
        # x-axis order: quantizations by descending median BPW
        _med_bpw = _fd.groupby(_qc)["bits_per_weight"].median().sort_values(ascending=False)
        _quant_order = _med_bpw.index.tolist()
        _q2pos = {q: i for i, q in enumerate(_quant_order)}

        _MARKER_POOL = ["o", "s", "^", "D", "v", "P", "X", "*", "h", "<"]
        _models = sorted(_fd["model_label"].unique())
        _mk_map = {m: _MARKER_POOL[i % len(_MARKER_POOL)] for i, m in enumerate(_models)}

        fig, ax = plt.subplots(figsize=(11, 5))

        for m in _models:
            _sub = _fd[_fd["model_label"] == m].copy()
            _sub["_xpos"] = _sub[_qc].map(_q2pos)
            _sub = _sub.sort_values("_xpos")
            ax.plot(_sub["_xpos"], _sub["fom_full"],
                    marker=_mk_map[m], ms=7, lw=2, label=m)

        # FoM = 1 reference
        ax.axhline(1.0, color="gray", ls="--", lw=1.2, alpha=0.7, label="Reference (FoM = 1)")

        # Mark usable configs with ★
        if "usable" in _fd.columns and _fd["usable"].any():
            _us = _fd[_fd["usable"]].copy()
            _us["_xpos"] = _us[_qc].map(_q2pos)
            ax.scatter(_us["_xpos"], _us["fom_full"],
                       s=220, marker="*", color="gold", zorder=5,
                       edgecolors="darkorange", lw=1, label="Usable (\u2265 T_hum \u2605)")

        # x labels: quantization name + median BPW
        _xticks = list(range(len(_quant_order)))
        _xlabels = [f"{q}\n({_med_bpw[q]:.1f} bpw)" for q in _quant_order]
        ax.set_xticks(_xticks)
        ax.set_xticklabels(_xlabels, fontsize=9)

        ax.set_xlabel("Quantization (median BPW)", fontsize=11)
        ax.set_ylabel("FoM_full (normalized vs reference)", fontsize=11)
        fig.suptitle("E1 — FoM_full vs quantization by model", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9, framealpha=0.9)
        fig.tight_layout()
        plt.show()

        _disp = [c for c in ["model_label", _qc, "bits_per_weight",
                              "fom_full", "tokens_per_s_mean", "usable"]
                 if c in fom_df.columns]
        print("FoM_full by model and quantization:")
        display(fom_df[_disp].sort_values(["model_label", "bits_per_weight"]).round(3))

except Exception:
    import traceback; traceback.print_exc()


In [ ]:
# K-quant vs non-K-quant comparison: throughput, perplexity, and J/token
_K_QUANTS     = {"Q2_K", "Q3_K_M", "Q4_K_M", "Q5_K_M", "Q6_K"}
_NON_K_QUANTS = {"Q4_0", "Q8_0"}
_FAM_COLORS   = {"K-quant": "#377eb8", "non-K": "#e41a1c"}

_q_col_k  = "quantization" if "quantization" in ex.columns else "model_short"
_ex_kfam  = ex.copy()
_ex_kfam["quant_family"] = _ex_kfam[_q_col_k].apply(
    lambda q: "K-quant" if str(q) in _K_QUANTS else ("non-K" if str(q) in _NON_K_QUANTS else None)
)
_ex_kfam = _ex_kfam.dropna(subset=["quant_family"])

if not _ex_kfam.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # (a) Throughput
    sns.boxplot(data=_ex_kfam, x="quant_family", y="tokens_per_s_mean",
                palette=_FAM_COLORS, showfliers=False, width=0.5, ax=axes[0])
    axes[0].set_title("(a) Throughput")
    axes[0].set_ylabel("tokens/s")
    axes[0].set_xlabel("")

    # (b) Perplejidad
    _ppl_kfam = _ex_kfam.dropna(subset=["perplexity_geomean"])
    if not _ppl_kfam.empty:
        sns.boxplot(data=_ppl_kfam, x="quant_family", y="perplexity_geomean",
                    palette=_FAM_COLORS, showfliers=False, width=0.5, ax=axes[1])
    axes[1].set_title("(b) Perplexity (geomean)")
    axes[1].set_ylabel("PPL")
    axes[1].set_xlabel("")

    # (c) J/token
    _ejt_kfam = _ex_kfam.dropna(subset=["energy_per_token_j"])
    if not _ejt_kfam.empty:
        sns.boxplot(data=_ejt_kfam, x="quant_family", y="energy_per_token_j",
                    palette=_FAM_COLORS, showfliers=False, width=0.5, ax=axes[2])
    axes[2].set_title("(c) Energy per token")
    axes[2].set_ylabel("J/token")
    axes[2].set_xlabel("")

    for ax in axes:
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle("E1 — K-quant vs non-K-quant: throughput, quality, and energy efficiency",
                 fontsize=12, y=1.02)
    fig.tight_layout()
    plt.show()

    print("\nSummary by quantization family:")
    display(_ex_kfam.groupby("quant_family")[
        ["tokens_per_s_mean", "perplexity_geomean", "energy_per_token_j"]
    ].agg(["median", "std"]).round(3))
else:
    print("No data for K-quant vs non-K comparison")


---
# 📎 APPENDIX — Supplementary material (not included in the main thesis)
---

The following figures are kept for validation and detail; they are not part of the 5 figures in the thesis.


## Executive summary
> TODO: Update with final results.


## Warning: DeepSeek + non-K-quants

> **Hypothesis to verify:** The `DeepSeek-R1-Distill-Qwen-1.5B` runs with **Q4_0 and Q8_0** quantizations produce invalid responses (ASCII character loop or empty string) instead of coherent text. Runs with K-quants (Q3_K_M, Q4_K_M, Q5_K_M) work correctly.

**Likely cause:** DeepSeek-R1 distilled models have weight distributions with pronounced outliers. **Non-K** quantizations (`Q4_0`, `Q8_0`) apply a uniform per-block scheme that destroys those outliers, causing generation collapse. **K-quants** (`_K_M`, `_K_S`) use mixed precision per layer and preserve the critical values.

**Impact on the analysis:**
- The inference metrics (tokens/s, perplexity) of Q4_0 and Q8_0 for DeepSeek are **invalid**.
- Consider **excluding them** from the E1 comparative analysis for that model.

**Pattern observed in the data:**
```
Q4_0 → response = '!"#$%&\'()*+,-./...'
Q8_0 → response = '' (empty) or 2-3 tokens
Q3_K_M / Q4_K_M / Q5_K_M → coherent response
```


In [ ]:
# ── Appendix: E1 comparative tables (inference + hardware with σ) ───
if not ex.empty:
    from monitorviz.transforms.fom import compute_fom_full, compute_usability

    _df_ann = ex.copy()

    # σ inference from pm_full per prompt
    if not pm_full.empty and "run_id" in pm_full.columns and "run_id" in _df_ann.columns:
        _valid_ann = (pm_full[~pm_full["is_empty_generation"]]
                      if "is_empty_generation" in pm_full.columns else pm_full)
        _si_ann = [c for c in ["tokens_per_second","time_to_first_token_ms","perplexity"]
                   if c in _valid_ann.columns]
        if _si_ann:
            _std_inf_ann = (_valid_ann.groupby("run_id")[_si_ann].std().reset_index()
                            .rename(columns={
                                "tokens_per_second":      "tokens_per_s_std",
                                "time_to_first_token_ms": "ttft_ms_std",
                                "perplexity":             "perplexity_std",
                            }))
            _df_ann = _df_ann.merge(_std_inf_ann, on="run_id", how="left")

    # σ hardware from hw_full per sample
    if not hw_full.empty and "run_id" in hw_full.columns and "run_id" in _df_ann.columns:
        _sh_ann = [c for c in ["temperature_c","internal_power_w","freq_mean_ghz"]
                   if c in hw_full.columns]
        if _sh_ann:
            _std_hw_ann = (hw_full.groupby("run_id")[_sh_ann].std().reset_index()
                           .rename(columns={
                               "temperature_c":   "temp_std_c",
                               "internal_power_w": "power_std_w",
                               "freq_mean_ghz":   "freq_std_ghz",
                           }))
            _df_ann = _df_ann.merge(_std_hw_ann, on="run_id", how="left")

    if "usable" in _df_ann.columns:
        _df_ann["usable"] = _df_ann["usable"].map({True: "✓", False: "✗"})

    # Table 1 — Inference with σ
    _sort_by = ["model_label", "bits_per_weight"] if "bits_per_weight" in _df_ann.columns else ["model_label"]
    _df_ann_sorted = _df_ann.sort_values(_sort_by).reset_index(drop=True)

    _inf_cols_ann = [c for c in [
        "model_label", "quantization", "engine",
        "tokens_per_s_mean",    "tokens_per_s_std",
        "ttft_ms_mean",         "ttft_ms_std",
        "latency_ms_mean",
        "perplexity_geomean",   "ppl_wk_stderr",  "perplexity_std",
    ] if c in _df_ann_sorted.columns]
    print("E1 comparative table — Inference")
    display(_df_ann_sorted[_inf_cols_ann].round(3))

    # Table 2 — Hardware with σ
    _hw_cols_ann = [c for c in [
        "model_label", "quantization", "engine",
        "temp_max_c",       "temp_mean_c",  "temp_std_c",
        "power_mean_w",     "power_std_w",
        "freq_std_ghz",
        "throttled_ratio",
        "energy_per_token_j", "cpu_work_core_s", "cpu_efficiency",
    ] if c in _df_ann_sorted.columns]
    print("\nE1 comparative table — Hardware")
    display(_df_ann_sorted[_hw_cols_ann].round(3))
else:
    print("No data.")


In [ ]:
# Temperature and power timelines (small multiples, 2 columns)
_run_ids = ex["run_id"].tolist() if "run_id" in ex.columns else []
hw_ex = hw_full[hw_full["run_id"].isin(_run_ids)] if "run_id" in hw_full.columns else pd.DataFrame()

# run_id → quantization map to color the lines
_q_col_tl = "quantization" if "quantization" in ex.columns else "model_short"
_rid_to_quant = ex.set_index("run_id")[_q_col_tl].to_dict() if "run_id" in ex.columns else {}

def _plot_timeline(hw_ex, models, y_col, ylabel, suptitle, hline=None):
    from matplotlib.lines import Line2D as _L2D
    ncols = 2
    nrows = (len(models) + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    quants_seen = []
    for ax, model in zip(axes, models):
        hw_m = hw_ex[hw_ex["model_label"] == model]
        if hw_m.empty:
            ax.text(0.5, 0.5, f"{model}\nNo data", ha="center", va="center",
                    transform=ax.transAxes)
            ax.set_axis_off()
            continue
        for rid in hw_m["run_id"].unique():
            sub = hw_m[hw_m["run_id"] == rid].sort_values("t_rel_s")
            q = _rid_to_quant.get(rid, "")
            color = QUANT_COLORS.get(q, None)
            if q and q not in quants_seen:
                quants_seen.append(q)
            ax.plot(sub["t_rel_s"].values, sub[y_col].values,
                    color=color, alpha=0.8, linewidth=1.5)
        if hline is not None:
            ax.axhline(hline, ls="--", color="red", alpha=0.5, label=f"Limit ({hline}°C)")
        total_s = hw_m["t_rel_s"].max()
        ax.set_xlabel("Time (min)" if total_s > 1800 else "Time (s)")
        ax.set_ylabel(ylabel)
        ax.set_title(model, fontweight="bold")
        ax.grid(True, alpha=0.3)
    for ax in axes[len(models):]:
        ax.set_visible(False)
    # Quantization legend below the title
    quants_ordered = [q for q in QUANT_ORDER if q in quants_seen]
    quants_ordered += [q for q in quants_seen if q not in quants_ordered]
    handles = [
        _L2D([0], [0], color=QUANT_COLORS.get(q, "gray"), lw=2, label=q)
        for q in quants_ordered
    ]
    if hline is not None:
        handles.append(_L2D([0], [0], color="red", lw=1.5, ls="--", label=f"Limit ({hline}°C)"))
    fig.legend(
        handles=handles, loc="upper center",
        ncol=len(handles), fontsize=9,
        title="Quantization", title_fontsize=9,
        bbox_to_anchor=(0.5, 1.0),
    )
    fig.suptitle(suptitle, fontsize=13, y=1.05)
    fig.tight_layout()
    plt.show()

if not hw_ex.empty:
    models = sorted(ex["model_label"].unique())
    _plot_timeline(hw_ex, models, "temperature_c", "Temperature (°C)",
                   "Temperature by model", hline=80)
    _plot_timeline(hw_ex, models, "internal_power_w", "Power (W)",
                   "Internal power by model")
else:
    print("No hardware data for timelines")


In [ ]:
# CPU, memory, and frequency distribution — small multiples by quantization
_run_ids = ex["run_id"].tolist() if "run_id" in ex.columns else []
hw_ex = hw_full[hw_full["run_id"].isin(_run_ids)].copy() if "run_id" in hw_full.columns else pd.DataFrame()

if not hw_ex.empty:
    hw_ex["quant_label"] = hw_ex["run_id"].map(_ql_map)
    _q_col_d  = "quantization" if "quantization" in ex.columns else "model_short"
    _rid_to_q = ex.set_index("run_id")[_q_col_d].to_dict()
    hw_ex["quantization"] = hw_ex["run_id"].map(_rid_to_q)

    # Order by ascending BPW (if available)
    if "bits_per_weight" in ex.columns:
        _bpw_map  = ex.set_index("quant_label")["bits_per_weight"].to_dict()
        _order_d  = sorted(
            hw_ex["quant_label"].dropna().unique(),
            key=lambda q: _bpw_map.get(q, 99),
        )
    else:
        _order_d = sorted(hw_ex["quant_label"].dropna().unique())

    # quant_label → quantization color
    _ql_color_d = {
        row["quant_label"]: QUANT_COLORS.get(str(row.get(_q_col_d, "")), "tab:gray")
        for _, row in ex.iterrows()
        if pd.notna(row.get("quant_label"))
    }

    def _hist_by_quant(hw_data, col, xlabel, suptitle,
                       xlim=None, vline=None, vline_label=None, cpu_filter=False):
        if col not in hw_data.columns:
            print(f"{col} not available"); return
        quants = [q for q in _order_d if q in hw_data["quant_label"].values]
        ncols  = 3
        nrows  = -(-len(quants) // ncols)
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(6 * ncols, 4 * nrows),
                                 sharey=False)
        axes_flat = axes.flatten() if nrows * ncols > 1 else [axes]
        for i, ql in enumerate(quants):
            ax   = axes_flat[i]
            data = hw_data[hw_data["quant_label"] == ql][col].dropna()
            if cpu_filter:
                data = data[data > 10]
            if data.empty:
                ax.set_visible(False); continue
            hist_range = xlim if xlim else (data.min(), data.max())
            if hist_range[0] == hist_range[1]:
                hist_range = (hist_range[0] - 0.5, hist_range[1] + 0.5)
            color = _ql_color_d.get(ql, "tab:gray")
            ax.hist(data, bins=30, color=color, alpha=0.75, edgecolor="white",
                    range=hist_range)
            ax.axvline(data.mean(), color="black", ls="--", lw=1.5,
                       label=f"μ={data.mean():.2f}")
            if vline is not None:
                ax.axvline(vline, color="gray", ls=":", lw=1.2,
                           label=vline_label or str(vline))
            ax.legend(fontsize=7)
            if xlim:
                ax.set_xlim(*xlim)
            ax.set_title(
                f"{ql}\nμ={data.mean():.2f}  σ={data.std():.2f}",
                fontsize=10,
            )
            ax.set_xlabel(xlabel)
            ax.set_ylabel("Samples")
        for j in range(len(quants), len(axes_flat)):
            axes_flat[j].set_visible(False)
        fig.suptitle(suptitle, fontsize=13, y=1.01)
        fig.tight_layout()
        plt.show()

    _hist_by_quant(hw_ex, "cpu_usage_pct", "CPU (%)",
                   "E1 — CPU distribution by quantization (samples with CPU > 10%)",
                   cpu_filter=True)
    _hist_by_quant(hw_ex, "mem_pct", "Memory (%)",
                   "E1 — Memory distribution by quantization")
    _hist_by_quant(hw_ex, "freq_mean_ghz", "Frequency (GHz)",
                   "E1 — Mean frequency distribution by quantization",
                   vline=2.4, vline_label="2.4 GHz nominal")
else:
    print("No hardware data for CPU/Memory/Frequency distributions")


In [ ]:
# Swap by model (small multiples)
_run_ids = ex["run_id"].tolist() if "run_id" in ex.columns else []
hw_ex = hw_full[hw_full["run_id"].isin(_run_ids)] if "run_id" in hw_full.columns else pd.DataFrame()

if not hw_ex.empty:
    models = sorted(ex["model_label"].unique())
    ncols = 2
    nrows = (len(models) + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, model in zip(axes, models):
        hw_m = hw_ex[hw_ex["model_label"] == model]
        if hw_m.empty:
            ax.text(0.5, 0.5, f"{model}\nNo data", ha="center", va="center",
                    transform=ax.transAxes)
            ax.set_axis_off()
            continue
        for rid in hw_m["run_id"].unique():
            sub = hw_m[hw_m["run_id"] == rid].sort_values("t_rel_s")
            ax.plot(sub["t_rel_s"].values, sub["swap_pct"].values, alpha=0.6, linewidth=1)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Swap (%)")
        ax.set_title(model, fontweight="bold")
        ax.grid(True, alpha=0.3)
    for ax in axes[len(models):]:
        ax.set_visible(False)
    fig.suptitle("Swap usage by model", fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("No hardware data for swap")

In [ ]:
# Detailed analysis of a representative run (median duration)
from monitorviz.viz import temp_freq_dual, cpu_memory_dual_phases, hw_distributions_panel

_run_ids = ex["run_id"].tolist() if "run_id" in ex.columns else []
hw_ex = hw_full[hw_full["run_id"].isin(_run_ids)] if "run_id" in hw_full.columns else pd.DataFrame()

if not hw_ex.empty and len(ex) > 0:
    run_durations = hw_ex.groupby("run_id")["t_rel_s"].max().sort_values()
    median_idx = len(run_durations) // 2
    representative_run_id = run_durations.index[median_idx]
    representative_run = next((r for r in coll.runs if r.run_id == representative_run_id), None)
    if representative_run:
        hw_r = hw_ex[hw_ex["run_id"] == representative_run_id]
        try:
            fig = temp_freq_dual(hw_r, representative_run)
            plt.show()
        except Exception as e:
            print(f"temp_freq_dual: {e}")
        try:
            fig = cpu_memory_dual_phases(hw_r, representative_run)
            plt.show()
        except Exception as e:
            print(f"cpu_memory_dual_phases: {e}")
        try:
            fig = hw_distributions_panel(hw_r, representative_run)
            plt.show()
        except Exception as e:
            print(f"hw_distributions_panel: {e}")
    else:
        print("No representative run found")
else:
    print("No data for detailed analysis")

In [ ]:
# Hardware grouped bar chart (4 panels)
_run_ids = ex["run_id"].tolist() if "run_id" in ex.columns else []
hw_ex = hw_full[hw_full["run_id"].isin(_run_ids)] if "run_id" in hw_full.columns else pd.DataFrame()

hw_summary_rows = []
pm_valid_tmp = (pm_full[
                    (pm_full["run_id"].isin(_run_ids)) &
                    (~pm_full["is_empty_generation"]) &
                    (pm_full["latency_ms"] > 0)
                    ] if "run_id" in pm_full.columns else pd.DataFrame())

_q_col_gb = "quantization" if "quantization" in ex.columns else "model_short"

for _, row in ex.iterrows():
    if "run_id" not in row.index:
        continue

    hw_r = hw_ex[hw_ex["run_id"] == row["run_id"]] if "run_id" in hw_ex.columns else pd.DataFrame()
    pm_r = pm_valid_tmp[pm_valid_tmp["run_id"] == row["run_id"]] if "run_id" in pm_valid_tmp.columns else pd.DataFrame()

    if hw_r.empty:
        continue

    active_r = hw_r[hw_r["cpu_usage_pct"] > 50]
    quant_name = str(row.get(_q_col_gb, "?"))

    hw_summary_rows.append({
        "model": row["model_label"],
        "config": quant_name,
        "cpu_pct": active_r["cpu_usage_pct"].mean() if not active_r.empty else float("nan"),
        "tokens_s": pm_r["tokens_per_second"].mean() if not pm_r.empty else float("nan"),
        "power_w": hw_r["internal_power_w"].mean(),
        "mem_mb": hw_r["mem_used_bytes"].mean() / (1024 ** 2) if not hw_r.empty else float("nan"),
    })

if hw_summary_rows:
    hw_summary = pd.DataFrame(hw_summary_rows)

    METRICS_GB = [
        ("cpu_pct", "Mean CPU (%)", "(a)"),
        ("tokens_s", "Throughput (tok/s)", "(b)"),
        ("power_w", "Power (W)", "(c)"),
        ("mem_mb", "Memory (MB)", "(d)"),
    ]

    models_gb = sorted(hw_summary["model"].unique())

    configs_gb = [q for q in QUANT_ORDER if q in hw_summary["config"].unique()]
    configs_gb += [q for q in hw_summary["config"].unique() if q not in QUANT_ORDER]

    x_pos = np.arange(len(models_gb))
    n_cfg = len(configs_gb)
    width = 0.75 / max(n_cfg, 1)

    fig, axes = plt.subplots(
        len(METRICS_GB),
        1,
        figsize=(12, 4 * len(METRICS_GB))
    )

    for i, (ax, (col, ylabel, label)) in enumerate(zip(axes, METRICS_GB)):
        data = hw_summary.dropna(subset=[col])

        for j, cfg in enumerate(configs_gb):
            color = QUANT_COLORS.get(cfg, f"C{j}")

            vals = [
                data[
                    (data["model"] == m) &
                    (data["config"] == cfg)
                    ][col].mean()
                for m in models_gb
            ]

            offset = (j - n_cfg / 2 + 0.5) * width

            ax.bar(
                x_pos + offset,
                vals,
                width * 0.9,
                label=cfg,
                color=color,
                alpha=0.85
            )

        ax.set_xticks(x_pos)
        ax.set_xticklabels(models_gb, rotation=45, ha="right")
        ax.set_ylabel(ylabel)

        ax.text(
            -0.06,
            1.03,
            label,
            transform=ax.transAxes,
            fontweight="bold",
            fontsize=12
        )

    # Get legend handles
    handles, labels = axes[0].get_legend_handles_labels()

    # Overall title
    fig.suptitle(
        "Hardware metrics by model and quantization",
        fontsize=13,
        y=0.99
    )

    # Global legend below the title
    fig.legend(
        handles,
        labels,
        title="Quantization",
        loc="upper center",
        bbox_to_anchor=(0.5, 0.965),
        ncol=min(len(configs_gb), 6),
        frameon=True
    )

    # Reserve space above for title and legend
    fig.tight_layout(rect=[0, 0, 1, 0.88])

    plt.show()

else:
    print("No data for hardware grouped bar chart")

In [ ]:
# Inference metrics by quantization
def _plot_metric_e1(df, metric, ylabel, title, unit_divisor=1.0, unit_label=""):
    data = df.dropna(subset=[metric]).copy()
    if "quant_label" not in data.columns:
        data["quant_label"] = data["run_id"].map(_ql_map)
    data["_val"] = data[metric] / unit_divisor
    order = (data.groupby("quant_label")["_val"].mean()
             .sort_values().index.tolist())
    means = data.groupby("quant_label")["_val"].mean().reindex(order)
    fig, ax = plt.subplots(figsize=(10, max(4, len(order) * 0.55)))
    ax.barh(range(len(order)), means.values, alpha=0.85, color="steelblue")
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=9)
    ax.set_xlabel(unit_label or metric)
    ax.set_title(f"{title}" + (f" ({unit_label})" if unit_label else ""))
    fig.tight_layout()
    plt.show()

_run_ids_pm = ex["run_id"].tolist() if "run_id" in ex.columns else []
pm_valid = (pm_full[
    (pm_full["run_id"].isin(_run_ids_pm)) &
    (~pm_full["is_empty_generation"]) &
    (pm_full["latency_ms"] > 0)
].copy() if "run_id" in pm_full.columns else pd.DataFrame())

if not pm_valid.empty:
    pm_valid["quant_label"] = pm_valid["run_id"].map(_ql_map)
    _plot_metric_e1(pm_valid, "tokens_per_second", "tokens/s",
                    "Throughput by quantization", unit_label="tokens/s")
    _plot_metric_e1(pm_valid, "words_per_second", "words/s",
                    "Words per second", unit_label="words/s")
    _plot_metric_e1(pm_valid, "latency_ms", "s", "Total latency per prompt",
                    unit_divisor=1000, unit_label="s")
    _plot_metric_e1(pm_valid, "time_to_first_token_ms", "ms",
                    "TTFT por prompt", unit_label="ms")
    ppl_valid = pm_valid.dropna(subset=["perplexity"])
    if not ppl_valid.empty:
        _plot_metric_e1(ppl_valid, "perplexity", "", "Perplexity per prompt",
                        unit_label="PPL")
else:
    print("No inference data")


In [ ]:
# Latency and perplexity ECDFs — one panel per model, lines by quantization
from matplotlib.lines import Line2D as _L2D

_run_ids_pm = ex["run_id"].tolist() if "run_id" in ex.columns else []
pm_valid = (pm_full[
    (pm_full["run_id"].isin(_run_ids_pm)) &
    (~pm_full["is_empty_generation"]) &
    (pm_full["latency_ms"] > 0)
].copy() if "run_id" in pm_full.columns else pd.DataFrame())

if pm_valid.empty:
    print("No data for ECDFs")
else:
    # Add only the columns missing from pm_valid (model_label already comes from pm_full)
    _cols_to_add = [c for c in ["model_label", "quantization"] if c not in pm_valid.columns]
    if _cols_to_add:
        _run_meta = ex[["run_id"] + _cols_to_add].copy()
        pm_valid = pm_valid.merge(_run_meta, on="run_id", how="left")

    _models_ecdf = sorted(pm_valid["model_label"].dropna().unique())
    _ncols = 3
    _nrows = (len(_models_ecdf) + _ncols - 1) // _ncols

    def _ecdf_by_model(pm_df, metric, xlabel, title):
        fig, axes = plt.subplots(
            _nrows, _ncols,
            figsize=(5 * _ncols, 4 * _nrows),
            sharey=True,
        )
        axes = np.array(axes).flatten()

        quants_in_fig = [q for q in QUANT_ORDER if q in pm_df["quantization"].values]

        for ax, model in zip(axes, _models_ecdf):
            sub = pm_df[pm_df["model_label"] == model].dropna(subset=[metric])
            ax.set_title(model, fontweight="bold", fontsize=9)
            ax.set_xlabel(xlabel, fontsize=8)
            ax.set_ylabel("F(x)", fontsize=8)
            ax.set_ylim(-0.02, 1.05)
            ax.grid(True, alpha=0.3)

            if sub.empty:
                ax.text(0.5, 0.5, "No data", ha="center", va="center",
                        transform=ax.transAxes, color="gray")
                continue

            sns.ecdfplot(
                data=sub,
                x=metric,
                hue="quantization",
                hue_order=quants_in_fig,
                palette=QUANT_COLORS,
                ax=ax,
                linewidth=1.8,
            )
            leg = ax.get_legend()
            if leg:
                leg.remove()

        for ax in axes[len(_models_ecdf):]:
            ax.set_visible(False)

        # Shared legend at the bottom of the figure
        handles = [
            _L2D([0], [0], color=QUANT_COLORS.get(q, "gray"), lw=2, label=q)
            for q in quants_in_fig
        ]
        fig.legend(
            handles=handles, loc="lower center",
            ncol=len(quants_in_fig), fontsize=9,
            title="Quantization", title_fontsize=9,
            bbox_to_anchor=(0.5, -0.04),
        )
        fig.suptitle(title, fontsize=13)
        fig.tight_layout()
        plt.show()

    _ecdf_by_model(pm_valid, "latency_ms",  "Latency (ms)",       "ECDF of latency by model")

    ppl_df = pm_valid.dropna(subset=["perplexity"])
    if not ppl_df.empty:
        _ecdf_by_model(ppl_df, "perplexity", "Perplexity (PPL)", "ECDF of perplexity by model")


In [ ]:
# Response length distribution (eval_count)
# Small multiples: one panel per model, KDE line by quantization
_run_ids_pm = ex["run_id"].tolist() if "run_id" in ex.columns else []
pm_valid = (pm_full[
    (pm_full["run_id"].isin(_run_ids_pm)) &
    (~pm_full["is_empty_generation"]) &
    (pm_full["latency_ms"] > 0)
].copy() if "run_id" in pm_full.columns else pd.DataFrame())

if not pm_valid.empty:
    _q_col_pm = "quantization" if "quantization" in pm_valid.columns else "model_short"
    # Add quantization if missing in pm_full
    if _q_col_pm not in pm_valid.columns or pm_valid[_q_col_pm].isna().all():
        _qmap = ex.set_index("run_id")[_q_col_s].to_dict()
        pm_valid[_q_col_pm] = pm_valid["run_id"].map(_qmap)

    _models_resp = sorted(pm_valid["model_label"].unique())
    _ncols = 3
    _nrows = (_len := len(_models_resp), (_len + _ncols - 1) // _ncols)[1]
    fig, axes = plt.subplots(_nrows, _ncols,
                             figsize=(5 * _ncols, 4 * _nrows),
                             sharey=False)
    axes = axes.flatten()

    for ax, model in zip(axes, _models_resp):
        sub_m = pm_valid[pm_valid["model_label"] == model]
        _quants_m = [q for q in QUANT_ORDER if q in sub_m[_q_col_pm].values]
        for q in _quants_m:
            data_q = sub_m[sub_m[_q_col_pm] == q]["eval_count"].dropna()
            if len(data_q) >= 5:
                sns.kdeplot(data_q, ax=ax,
                            color=QUANT_COLORS.get(q, "gray"),
                            label=q, linewidth=1.8, fill=True, alpha=0.15)
            elif len(data_q) > 0:
                ax.axvline(data_q.mean(), color=QUANT_COLORS.get(q, "gray"),
                           lw=2, ls="--", label=f"{q} (n={len(data_q)})")
        ax.set_title(model, fontweight="bold", fontsize=9)
        ax.set_xlabel("Generated tokens (eval_count)", fontsize=8)
        ax.set_ylabel("Density", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7, ncol=2)

    for ax in axes[len(_models_resp):]:
        ax.set_visible(False)

    fig.suptitle("E1 — Response length distribution by model and quantization",
                 fontsize=12, y=1.01)
    fig.tight_layout()
    plt.show()

    # Summary table: median ± std per model (more readable than by quant_label)
    _summary_resp = pm_valid.groupby(["model_label", _q_col_pm])["eval_count"].agg(
        N="count", Median="median", Mean="mean", Std="std", Min="min", Max="max"
    ).round(1)
    _summary_resp.index.names = ["Model", "Quant."]
    display(_summary_resp)
else:
    print("No data for response length distribution")


In [ ]:
# Sensitivity curve BPW — one figure per model
from monitorviz.viz import sensitivity_curve

_SENS_METRICS = [
    ("tokens_per_s_mean",  "Throughput (tok/s)"),
    ("perplexity_geomean", "Perplexity"),
    ("energy_per_token_j", "Energy (J/tok)"),
    ("latency_ms_mean",    "Latency (ms)"),
]

e1_ok = (not ex.empty
         and "bits_per_weight" in ex.columns
         and ex["bits_per_weight"].nunique() >= 2)

if e1_ok:
    _models_sc = sorted(ex["model_label"].dropna().unique())
    for _m in _models_sc:
        _sub = ex[ex["model_label"] == _m].copy()
        if _sub["bits_per_weight"].nunique() < 2:
            continue
        fig = sensitivity_curve(
            _sub,
            param_col="bits_per_weight",
            metrics=_SENS_METRICS,
            title=f"Sensitivity BPW — {_m}",
            param_label="BPW",
        )
        plt.show()

    if "model_size_gb" in ex.columns:
        size_bpw = (ex.dropna(subset=["bits_per_weight", "model_size_gb"])
                    .groupby("bits_per_weight")["model_size_gb"].mean())
        if not size_bpw.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(size_bpw.index, size_bpw.values, marker="o", linewidth=2)
            ax.set_xlabel("BPW")
            ax.set_ylabel("Size (GBytes)")
            ax.set_title("Model size vs BPW")
            ax.grid(True, alpha=0.3)
            fig.tight_layout()
            plt.show()
else:
    print("Insufficient BPW data")


In [ ]:
# MBU_corr — Memory Bandwidth Utilization (architecture-corrected)
from monitorviz.viz.style import TARGET_PEAK_MEMORY_BANDWIDTH_GBs

_mbu_col = "mbu_corr" if "mbu_corr" in ex.columns else (
           "mbu_pct"  if "mbu_pct"  in ex.columns else None)

if _mbu_col is not None:
    mbu_data = ex.dropna(subset=[_mbu_col]).copy()
    if not mbu_data.empty:
        has_fan = "fan" in mbu_data.columns and mbu_data["fan"].nunique() > 1
        fan_groups = (
            [(True, "With fan (FAN)", "steelblue"),
             (False, "Without fan (NOFAN)", "tomato")]
            if has_fan else [(None, "MBU_corr", "steelblue")]
        )
        fig, axes = plt.subplots(1, len(fan_groups),
                                  figsize=(7 * len(fan_groups), 5), sharey=False)
        if len(fan_groups) == 1:
            axes = [axes]
        for ax, (fan_val, lbl, color) in zip(axes, fan_groups):
            sub = (mbu_data[mbu_data["fan"] == fan_val]
                   if fan_val is not None else mbu_data).copy()
            sub = sub.sort_values(_mbu_col)
            if sub.empty:
                ax.set_axis_off(); continue
            ax.barh(sub["model_label"], sub[_mbu_col], color=color, alpha=0.85)
            ax.axvline(100, ls="--", color="gray", alpha=0.5)
            ax.set_xlabel("MBU_corr (%)")
            ax.set_title(lbl)
            ax.grid(True, axis="x", alpha=0.3)
            for bar, val in zip(ax.patches, sub[_mbu_col]):
                ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                        f"{val:.1f}%", va="center", fontsize=9)
        fig.suptitle(
            f"E1 — MBU_corr: actual bandwidth utilization\n"
            f"(Peak BW RPi5: {TARGET_PEAK_MEMORY_BANDWIDTH_GBs} GB/s, LPDDR4X-4267, 32-bit bus)\n"
            "† gemma3n: upper bound (selective activation, actual MBU_corr < theoretical)",
            fontsize=10)
        plt.tight_layout()
        plt.show()
        if "mbu_corr_notes" in mbu_data.columns:
            notes = (mbu_data[["model_label","mbu_corr","mbu_corr_notes"]]
                     .drop_duplicates("model_label").round(2))
            print("Corrections applied:")
            display(notes)
else:
    print("mbu_corr not available in this experiment.")


In [ ]:
# Multidimensional trade-offs — Pareto frontier
def _pareto_front(df, x_col, y_col, x_lower, y_lower):
    pts = df[[x_col, y_col]].to_numpy(dtype=float)
    n = len(pts)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            x_ok = (pts[j, 0] <= pts[i, 0]) if x_lower else (pts[j, 0] >= pts[i, 0])
            y_ok = (pts[j, 1] <= pts[i, 1]) if y_lower else (pts[j, 1] >= pts[i, 1])
            strict = (pts[j, 0] != pts[i, 0]) or (pts[j, 1] != pts[i, 1])
            if x_ok and y_ok and strict:
                dominated[i] = True
                break
    return df[~dominated].sort_values(x_col)


pareto_pairs = [
    ("tokens_per_s_mean",  "energy_per_token_j",  "Throughput (tok/s)", "Energy/token (J)", False, True),
   # ("perplexity_geomean", "tokens_per_s_mean",   "Perplexity",         "Throughput (tok/s)", True,  False),
    #("energy_per_token_j", "perplexity_geomean",  "Energy/token (J)",   "Perplexity",         True,  True),
]
valid_pairs = [
    p for p in pareto_pairs
    if p[0] in ex.columns and p[1] in ex.columns
]

if valid_pairs and not ex.empty:
    _q_col_p = "quantization" if "quantization" in ex.columns else "model_short"
    _quants_p = [q for q in QUANT_ORDER if q in ex[_q_col_p].values]
    _models_p = sorted(ex["model_label"].unique())

    ncols = len(valid_pairs)
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
    if ncols == 1:
        axes = [axes]

    for ax, (x_col, y_col, xlabel, ylabel, x_lower, y_lower) in zip(axes, valid_pairs):
        data = ex.dropna(subset=[x_col, y_col]).reset_index(drop=True)
        if data.empty:
            ax.set_visible(False)
            continue

        scatter_e1(ax, data, x_col, y_col,
                   xlabel + (" ↓ better" if x_lower else " ↑ better"),
                   ylabel + (" ↓ better" if y_lower else " ↑ better"),
                   f"{xlabel} vs {ylabel}")

        front = _pareto_front(data, x_col, y_col, x_lower, y_lower)
        if not front.empty:
            ax.plot(front[x_col].values, front[y_col].values,
                    color="tab:red", ls="--", lw=1.5, zorder=2, label="Pareto frontier")

    # Shared legend outside top: color=quantization, shape=model
    _quant_handles = [
        _Line2D([0], [0], marker="o", color="w",
                markerfacecolor=QUANT_COLORS.get(q, "gray"), markersize=9, label=q)
        for q in _quants_p
    ]
    _model_handles = [
        _Line2D([0], [0], marker=MODEL_MARKERS.get(m, "o"), color="gray",
                markersize=9, linestyle="None", label=m)
        for m in _models_p
    ]
    _pareto_h = _Line2D([0], [0], color="tab:red", ls="--", lw=1.5, label="Pareto frontier")
    fig.legend(
        handles=_quant_handles + _model_handles + [_pareto_h],
        loc="upper center", bbox_to_anchor=(0.5, 1.06),
        ncol=min(len(_quants_p) + len(_models_p) + 1, 6),
        fontsize=9, title="● Color=Quantization   ▲ Shape=Model"
    )
    fig.suptitle("E1 — Multidimensional trade-offs (Pareto frontier)", fontsize=13, y=1.16)
    fig.tight_layout()
    plt.show()
else:
    print("Insufficient columns for Pareto:", [p[0] for p in pareto_pairs])


In [ ]:
# Phase breakdown: prefill vs decode
# Uses prompt_eval_duration_ns (prefill) and eval_duration_ns (decode) from pm_full
_run_ids_ph = ex["run_id"].tolist() if "run_id" in ex.columns else []
pm_ph = (pm_full[
    (pm_full["run_id"].isin(_run_ids_ph)) &
    (~pm_full["is_empty_generation"]) &
    (pm_full["latency_ms"] > 0)
].copy() if "run_id" in pm_full.columns else pd.DataFrame())

if not pm_ph.empty and "prompt_eval_duration_ns" in pm_ph.columns and "eval_duration_ns" in pm_ph.columns:
    pm_ph["phase_prefill_s"] = pm_ph["prompt_eval_duration_ns"] / 1e9
    pm_ph["phase_decode_s"]  = pm_ph["eval_duration_ns"] / 1e9
    pm_ph["quant_label"]     = pm_ph["run_id"].map(_ql_map)

    phase_agg = pm_ph.groupby("quant_label")[["phase_prefill_s", "phase_decode_s"]].median()
    phase_agg["_total"] = phase_agg.sum(axis=1)
    phase_agg = phase_agg.sort_values("_total").drop(columns="_total")
    phase_norm = phase_agg.div(phase_agg.sum(axis=1), axis=0) * 100

    _COLORS_PH = {"phase_prefill_s": "#377eb8", "phase_decode_s": "#e41a1c"}
    _LABELS_PH = {"phase_prefill_s": "Prefill (prompt eval)", "phase_decode_s": "Decode (generation)"}

    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(phase_agg) * 0.55)))
    for ax, data, title, unit in [
        (axes[0], phase_norm, "Phase breakdown — normalized (%)", "%"),
        (axes[1], phase_agg,  "Phase breakdown — absolute (s)",    "s"),
    ]:
        bottom = np.zeros(len(data))
        x = np.arange(len(data))
        for col in ["phase_prefill_s", "phase_decode_s"]:
            ax.barh(x, data[col].values, left=bottom,
                    color=_COLORS_PH[col], label=_LABELS_PH[col], alpha=0.85)
            bottom += data[col].values
        ax.set_yticks(x)
        ax.set_yticklabels(data.index, fontsize=8)
        ax.set_xlabel(unit)
        ax.set_title(title, fontweight="bold")
        ax.legend(loc="lower right", fontsize=9)
        ax.grid(True, alpha=0.3, axis="x")

    fig.suptitle("E1 — Phase breakdown: prefill vs decode (median by quantization)",
                 fontsize=12, y=1.02)
    fig.tight_layout()
    plt.show()

    phase_agg["prefill_%"] = (phase_agg["phase_prefill_s"] / phase_agg.sum(axis=1) * 100).round(1)
    phase_agg["decode_%"]  = (phase_agg["phase_decode_s"]  / phase_agg.sum(axis=1) * 100).round(1)
    display(phase_agg[["phase_prefill_s", "phase_decode_s", "prefill_%", "decode_%"]].round(3))
else:
    print("No phase data (prompt_eval_duration_ns / eval_duration_ns not available)")


In [ ]:
# Throttling heatmap
try:
    from monitorviz.viz import throttling_heatmap
    _run_ids_16 = ex["run_id"].tolist() if "run_id" in ex.columns else []
    hw_ex = hw_full[hw_full["run_id"].isin(_run_ids_16)] if "run_id" in hw_full.columns else pd.DataFrame()
    runs_ex = [r for r in coll.runs if r.run_id in _run_ids_16]
    if hw_ex.empty or not runs_ex:
        print("No hardware data for throttling heatmap")
    else:
        fig = throttling_heatmap(hw_ex, runs_ex, bins=40,
                                 title="Throttling heatmap")
        plt.show()
except Exception as e:
    print(f"Error in throttling_heatmap: {e}")

## Computational efficiency: effective CPU work

Standard metrics (tokens/s, MBU) don't distinguish between "fast because it's a good model" and "fast but the CPU is at 50% due to memory-bound behavior". The following three metrics capture **effective CPU compute**, adjusted for throttling.

---

### Metric 1 — Effective CPU work (W_CPU)

Analogous to person-hours. Measures how many "CPU-seconds-at-nominal-frequency" have been consumed:

$$
W_{CPU} = \int_0^T \frac{f(t)}{f_{nom}} \cdot \frac{U(t)}{100} \cdot N_{cores} \; dt
$$

| Symbol | Meaning |
|---------|-------------|
| $f(t)$ | Instantaneous frequency (GHz), per hw sample |
| $f_{nom}$ | SoC nominal frequency — **2.4 GHz** on RPi 5 |
| $U(t)$ | CPU usage percentage |
| $N_{cores}$ | Number of cores — **4** on RPi 5 |

**Units**: core·s equivalent at nominal frequency.

> If a run lasts 100 s and all 4 cores work at 100% at nominal frequency, $W_{CPU} = 400$ core·s.
> With sustained throttling at 1.2 GHz (50% of nominal), $W_{CPU} = 200$ core·s — half the effective work.

Discrete implementation over `hw_metrics_df`:

```python
def cpu_work_effective(hw_df, n_cores=4, f_nom_ghz=2.4, period_s=0.5):
    return (
        (hw_df["freq_mean_ghz"] / f_nom_ghz) *
        (hw_df["cpu_usage_pct"] / 100) *
        n_cores *
        period_s
    ).sum()
```

---

### Metric 2 — CPU efficiency (η_CPU)

Ratio between the effective work performed and the theoretical maximum possible during the run:

$$
\eta_{CPU} = \frac{W_{CPU}}{N_{cores} \cdot T}
$$

**Units**: dimensionless, $\eta \in [0, 1]$.

> $\eta = 1.0$: all 4 cores ran at 100% at nominal frequency for the entire run.
> $\eta = 0.5$: half of the available compute was used (throttling, idle time, memory contention, etc.).

---

### Metric 3 — Work per token (W_token)

Normalizes the effective work by generated tokens — a *compute-aware* analog of per-token latency:

$$
W_{token} = \frac{W_{CPU}}{N_{tokens}} \quad [\text{core·s/token}]
$$

---

### Why these are useful in this thesis

| Existing metric | What it measures | What it **doesn't** measure |
|-------------------|-------------|---------------------|
| tokens/s | Raw throughput | Whether the CPU is throttled or idle |
| MBU | Memory bus saturation | CPU work |
| **W_CPU** | Effective CPU compute adjusted for throttling | — |
| **η_CPU** | Fraction of theoretical compute used | — |
| **W_token** | Computational cost per generated token | — |

The `ministral` run, throttled 99% of the time at ~1.0 GHz, has a much lower $W_{CPU}$ than a hypothetical run at a sustained 2.4 GHz. This metric quantifies the **exact difference**, and comparing across quantizations reveals whether the heavier variants (Q8_0, BF16) spend proportionally more CPU compute per generated token.


In [ ]:
# Computational and energy efficiency
def _plot_eff_e1(df, metric, title, unit_label=""):
    data = df.dropna(subset=[metric]).copy()
    order = (data.groupby("quant_label")[metric].mean()
             .sort_values().index.tolist())
    means = data.groupby("quant_label")[metric].mean().reindex(order)
    fig, ax = plt.subplots(figsize=(10, max(4, len(order) * 0.55)))
    ax.barh(range(len(order)), means.values, alpha=0.85, color="steelblue")
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=9)
    ax.set_xlabel(unit_label or metric)
    ax.set_title(f"{title}" + (f" ({unit_label})" if unit_label else ""))
    fig.tight_layout()
    plt.show()

for col, ylabel, title in [
    ("cpu_work_core_s",    "core·s",       "Effective CPU work per run"),
    ("cpu_efficiency",     "",             "CPU efficiency eta_CPU"),
    ("cpu_work_per_token", "core·s/token", "CPU work per generated token"),
]:
    if col in ex.columns:
        _plot_eff_e1(ex, col, title=title, unit_label=ylabel)

# Phase breakdown
try:
    phase_df = coll.cpu_work_by_phase_df() if hasattr(coll, "cpu_work_by_phase_df") else pd.DataFrame()
    phase_df_ex = phase_df[phase_df["run_id"].isin(ex["run_id"])] if not phase_df.empty else pd.DataFrame()
    if not phase_df_ex.empty:
        phase_df_ex = phase_df_ex.copy()
        phase_df_ex["quant_label"] = phase_df_ex["run_id"].map(_ql_map)
        fig, ax = plt.subplots(figsize=(10, max(4, phase_df_ex["quant_label"].nunique() * 0.55)))
        phase_df_ex.groupby(["quant_label", "phase"])["cpu_work_core_s"].mean().unstack().plot(
            kind="barh", ax=ax)
        ax.set_title("W_CPU by phase and quantization")
        ax.set_ylabel("")
        fig.tight_layout()
        plt.show()
except Exception as e:
    print(f"cpu_work_by_phase_df: {e}")

if "energy_per_token_j" in ex.columns:
    _plot_eff_e1(ex, "energy_per_token_j", title="Energy per token", unit_label="J/token")

# CPU vs Power scatter
if "cpu_work_core_s" in ex.columns and "power_mean_w" in ex.columns:
    valid_data = ex.dropna(subset=["cpu_work_core_s", "power_mean_w"])
    if not valid_data.empty:
        _q_col_e = "quantization" if "quantization" in valid_data.columns else "model_short"
        fig, ax = plt.subplots(figsize=(12, 6))
        scatter_e1(ax, valid_data, "cpu_work_core_s", "power_mean_w",
                   "W_CPU (core·s)", "Mean power (W)", "W_CPU — Power relationship")
        _quants_e = [q for q in QUANT_ORDER if q in valid_data[_q_col_e].values]
        _models_e = sorted(valid_data["model_label"].unique())
        add_e1_legend(ax, _quants_e, _models_e)
        fig.tight_layout()
        plt.show()


In [ ]:
# Performance vs BPW
if "bits_per_weight" in ex.columns and ex["bits_per_weight"].notna().any():
    bpw_data = ex.dropna(subset=["bits_per_weight"])
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    scatter_e1(axes[0], bpw_data, "bits_per_weight", "tokens_per_s_mean",
               "BPW", "tokens/s", "Throughput vs BPW")

    ppl_bpw = bpw_data.dropna(subset=["perplexity_geomean"])
    if not ppl_bpw.empty:
        scatter_e1(axes[1], ppl_bpw, "bits_per_weight", "perplexity_geomean",
                   "BPW", "Perplexity (geomean)", "Perplexity vs BPW")
    else:
        axes[1].set_visible(False)

    _quants_s = [q for q in QUANT_ORDER if q in bpw_data[_q_col_s].values]
    _models_s = sorted(bpw_data["model_label"].unique())
    _legend_ax = axes[1] if not ppl_bpw.empty else axes[0]
    add_e1_legend(_legend_ax, _quants_s, _models_s)

    fig.suptitle("Performance vs quantization (BPW)", y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("No BPW data for scatter")


In [ ]:
# BPW vs η_CPU scatter (computational efficiency)
if "bits_per_weight" in ex.columns and "cpu_efficiency" in ex.columns:
    _bpw_eta = ex.dropna(subset=["bits_per_weight", "cpu_efficiency"]).copy()
    if not _bpw_eta.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        scatter_e1(ax, _bpw_eta, "bits_per_weight", "cpu_efficiency",
                   "BPW", "η_CPU", "CPU efficiency vs BPW")
        ax.set_ylim(0, 1.05)
        ax.axhline(1.0, ls="--", color="gray", alpha=0.5, label="η=1.0 (theoretical optimum)")
        ax.legend(loc="lower right", fontsize=9)

        _quants_eta = [q for q in QUANT_ORDER if q in _bpw_eta[_q_col_s].values]
        _models_eta = sorted(_bpw_eta["model_label"].unique())
        add_e1_legend(ax, _quants_eta, _models_eta)

        fig.suptitle("E1 — CPU efficiency (η_CPU) vs quantization (BPW)", y=1.02)
        fig.tight_layout()
        plt.show()
    else:
        print("No data for BPW vs η_CPU")
else:
    print("Columns bits_per_weight / cpu_efficiency not available")


In [ ]:
# Polar diagrams — pair (performance + hardware) by quantization
from matplotlib.lines import Line2D as _L2D

PERF_METRICS = [
    ("tokens_per_s_mean",  "Throughput\n(tok/s)", True),
    ("ttft_ms_mean",       "TTFT",                False),
    ("perplexity_geomean", "Perplexity",           False),
    ("energy_per_token_j", "Energy/token",         False),
]
HW_METRICS = [
    ("mbu_corr",       "MBU_corr (%)",   True),
    ("temp_max_c",     "Max temp", False),
    ("power_mean_w",   "Power",     False),
    ("cpu_efficiency", "η_CPU",     True),
]

_q_col_r = "quantization" if "quantization" in ex.columns else "model_short"

# Colors by model (active palette cycle)
_model_list_r   = sorted(ex["model_label"].dropna().unique())
_color_cycle_r  = plt.rcParams["axes.prop_cycle"].by_key()["color"]
MODEL_COLORS_R  = {m: _color_cycle_r[i % len(_color_cycle_r)]
                   for i, m in enumerate(_model_list_r)}


def _radar_axes(ax, avail_metrics, title):
    """Configures the common polar axes."""
    N      = len(avail_metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles)
    ax.set_xticklabels([m[1] for m in avail_metrics], size=8)
    ax.set_ylim(0, 1)
    ax.grid(color="gray", alpha=0.3)
    ax.set_title(title, pad=15, fontweight="bold", fontsize=10)
    return angles


def _norm_vals(row, avail_metrics):
    """Returns a list of normalized [0,1] values for a row."""
    vals = []
    for col, _, higher_is_better in avail_metrics:
        col_min = ex[col].min() if col in ex.columns else 0.0
        col_max = ex[col].max() if col in ex.columns else 1.0
        rng = col_max - col_min if col_max != col_min else 1.0
        raw = row.get(col, np.nan)
        v   = 0.0 if pd.isna(raw) else np.clip((raw - col_min) / rng, 0.0, 1.0)
        vals.append(v if higher_is_better else 1.0 - v)
    return vals


def _radar_fig(metrics, title, ex_df):
    """One panel per model; lines colored by quantization."""
    avail  = [m for m in metrics if m[0] in ex_df.columns]
    models = sorted(ex_df["model_label"].dropna().unique())
    if not models or not avail:
        print(f"No data for: {title}"); return
    ncols = min(len(models), 4)
    nrows = (len(models) + ncols - 1) // ncols
    fig   = plt.figure(figsize=(5 * ncols, 5 * nrows + 1))
    for k, model in enumerate(models):
        ax     = fig.add_subplot(nrows, ncols, k + 1, polar=True)
        subset = ex_df[ex_df["model_label"] == model]
        if subset.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off(); continue
        angles = _radar_axes(ax, avail, model)
        angles_c = angles + angles[:1]
        for _, row in subset.iterrows():
            vals  = _norm_vals(row, avail)
            color = QUANT_COLORS.get(str(row.get(_q_col_r, "")), "gray")
            ax.plot(angles_c, vals + vals[:1], lw=2, color=color, alpha=0.9)
            ax.fill(angles_c, vals + vals[:1], alpha=0.12, color=color)
    quants_in_fig = [q for q in QUANT_ORDER if q in ex_df[_q_col_r].values]
    quants_in_fig += [q for q in ex_df[_q_col_r].dropna().unique() if q not in quants_in_fig]
    handles = [_L2D([0],[0], color=QUANT_COLORS.get(q,"gray"), lw=2, label=q)
               for q in quants_in_fig]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles),
               fontsize=9, title="Quantization", title_fontsize=9,
               bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(title + "  (outer = better)", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()


def _radar_fig_t(metrics, title, ex_df):
    """Transposed version: one panel per quantization; lines colored by model."""
    avail  = [m for m in metrics if m[0] in ex_df.columns]
    quants = [q for q in QUANT_ORDER if q in ex_df[_q_col_r].values]
    quants += [q for q in ex_df[_q_col_r].dropna().unique() if q not in quants]
    if not quants or not avail:
        print(f"No data for: {title}"); return
    ncols = min(len(quants), 4)
    nrows = (len(quants) + ncols - 1) // ncols
    fig   = plt.figure(figsize=(5 * ncols, 5 * nrows + 1))
    for k, q in enumerate(quants):
        ax     = fig.add_subplot(nrows, ncols, k + 1, polar=True)
        subset = ex_df[ex_df[_q_col_r] == q]
        if subset.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off(); continue
        angles = _radar_axes(ax, avail, q)
        angles_c = angles + angles[:1]
        for _, row in subset.iterrows():
            vals  = _norm_vals(row, avail)
            color = MODEL_COLORS_R.get(str(row.get("model_label", "")), "gray")
            ax.plot(angles_c, vals + vals[:1], lw=2, color=color, alpha=0.9)
            ax.fill(angles_c, vals + vals[:1], alpha=0.12, color=color)
    handles = [_L2D([0],[0], color=MODEL_COLORS_R.get(m,"gray"), lw=2, label=m)
               for m in _model_list_r if m in ex_df["model_label"].values]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 4),
               fontsize=9, title="Model", title_fontsize=9,
               bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(title + "  (outer = better)", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()


# ── Version A: panel by model, lines by quantization ─────────────────────────
_radar_fig(PERF_METRICS, "Performance — panel by model",     ex)
_radar_fig(HW_METRICS,   "Hardware    — panel by model",     ex)

# ── Version B (transposed): panel by quantization, lines by model ────────────
_radar_fig_t(PERF_METRICS, "Performance — panel by quantization",  ex)
_radar_fig_t(HW_METRICS,   "Hardware    — panel by quantization",  ex)


In [ ]:
# Pearson correlation with BPW as the first variable
PEARSON_VARS = [
    "bits_per_weight",
    "tokens_per_s_mean", "words_per_s_mean", "latency_ms_mean",
    "ttft_ms_mean", "eval_count_mean",
    "perplexity_geomean",
    "temp_max_c", "power_mean_w", "energy_per_token_j", "cpu_work_core_s",
]
BLOCK_SIZES = [1, 6, 1, 4]
PEARSON_LABELS = {
    "bits_per_weight":    "BPW",
    "tokens_per_s_mean":  "tok/s",
    "words_per_s_mean":   "words/s",
    "latency_ms_mean":    "Latency",
    "ttft_ms_mean":       "TTFT",
    "eval_count_mean":    "Gen. tokens",
    "perplexity_geomean": "Perplexity",
    "temp_max_c":         "Max temp",
    "power_mean_w":       "Power",
    "energy_per_token_j": "J/token",
    "cpu_work_core_s":    "W_CPU",
}

if len(ex) >= 3:
    avail = [v for v in PEARSON_VARS if v in ex.columns]
    if len(avail) >= 2:
        corr_data = ex[avail].dropna(how="all")
        corr_mat  = corr_data.corr(method="pearson")
        labels    = [PEARSON_LABELS.get(v, v) for v in avail]

        fig, ax = plt.subplots(figsize=(max(8, len(avail) * 0.75),
                                        max(7, len(avail) * 0.65)))
        im = ax.imshow(corr_mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xticks(range(len(avail)))
        ax.set_yticks(range(len(avail)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
        ax.set_yticklabels(labels, fontsize=9)
        ax.grid(False)  # avoids grid lines over the heatmap
        for i in range(len(avail)):
            for j in range(len(avail)):
                v = corr_mat.iloc[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=7, color="white" if abs(v) > 0.6 else "black")

        # Block separators
        cum = 0
        for size in BLOCK_SIZES:
            cum += size
            if cum < len(avail):
                ax.axhline(cum - 0.5, color="white", lw=2)
                ax.axvline(cum - 0.5, color="white", lw=2)

        ax.set_title(f"Pearson correlation (n={len(corr_data)} runs)", fontsize=12)
        fig.tight_layout()
        plt.show()

        # Table: correlations with BPW
        if "bits_per_weight" in avail:
            bpw_corr = (corr_mat["bits_per_weight"]
                        .drop("bits_per_weight")
                        .sort_values(key=abs, ascending=False))
            bpw_df = bpw_corr.reset_index()
            bpw_df.columns = ["Variable", "Corr. with BPW"]
            bpw_df["Variable"] = bpw_df["Variable"].map(
                lambda x: PEARSON_LABELS.get(x, x))
            bpw_df["Corr. with BPW"] = bpw_df["Corr. with BPW"].round(3)
            print("\nCorrelations with bits_per_weight:")
            display(bpw_df)
    else:
        print(f"Insufficient variables: {avail}")
else:
    print(f"Insufficient data: {len(ex)} run(s), need >=3.")


In [ ]:
# Alerts and anomalies
alerts = []
if "throttled_ratio" in ex.columns:
    for _, row in ex[ex["throttled_ratio"] > 0.5].iterrows():
        alerts.append(f"throttling: {row['model_label']} ({row['throttled_ratio']*100:.1f}%)")
if "n_empty_generations" in ex.columns:
    for _, row in ex[ex["n_empty_generations"] > 0].iterrows():
        alerts.append(f"empty generations: {row['model_label']} (n={int(row['n_empty_generations'])})")
if "tokens_per_s_mean" in ex.columns and len(ex) >= 3:
    mean_tps = ex["tokens_per_s_mean"].mean()
    std_tps  = ex["tokens_per_s_mean"].std()
    for _, row in ex[(ex["tokens_per_s_mean"] > mean_tps + 2 * std_tps) |
                     (ex["tokens_per_s_mean"] < mean_tps - 2 * std_tps)].iterrows():
        alerts.append(f"throughput outlier: {row['model_label']} ({row['tokens_per_s_mean']:.2f} tok/s)")
for a in alerts:
    print(f"ALERT: {a}")
if not alerts:
    print("No alerts detected")

In [ ]:
# E1 speedup table — pairwise comparison of quantizations
if not ex.empty and "tokens_per_s_mean" in ex.columns:
    _q_col_sp = "quantization" if "quantization" in ex.columns else "model_short"
    _df_sp = ex.dropna(subset=["tokens_per_s_mean"]).copy()
    _df_sp["_config"] = _df_sp.get("quant_label", _df_sp["model_label"]).fillna("?")

    # Median per config if there are multiple runs
    _speed  = _df_sp.groupby("_config")["tokens_per_s_mean"].median()
    # Sort by BPW if available
    if "bits_per_weight" in _df_sp.columns:
        _bpw_map = _df_sp.groupby("_config")["bits_per_weight"].median()
        _labels  = _bpw_map.reindex(_speed.index).sort_values().index.tolist()
    else:
        _labels = _speed.index.tolist()
    _speed  = _speed.reindex(_labels)
    _n      = len(_labels)

    _mat     = np.zeros((_n, _n))
    for _i, _base in enumerate(_labels):
        for _j, _tgt in enumerate(_labels):
            _mat[_i, _j] = _speed[_tgt] / _speed[_base]

    _log_mat  = np.log2(_mat)
    _off_diag = _log_mat[~np.eye(_n, dtype=bool)]
    _abs_max  = max(float(abs(_off_diag).max()), 0.5)

    fig, ax = plt.subplots(figsize=(max(8, _n * 0.9), max(7, _n * 0.8)))
    im = ax.imshow(_log_mat, cmap="RdYlGn", vmin=-_abs_max, vmax=_abs_max,
                   aspect="auto")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("log₂(speedup)  [>0 = slower row]", fontsize=9)

    ax.set_xticks(range(_n)); ax.set_yticks(range(_n))
    ax.set_xticklabels(_labels, rotation=40, ha="right", fontsize=8)
    ax.set_yticklabels(_labels, fontsize=8)
    ax.set_xlabel("Compared quantization (faster → green)", fontsize=9)
    ax.set_ylabel("Base quantization (reference)", fontsize=9)
    ax.grid(False)

    for _i in range(_n):
        for _j in range(_n):
            _v  = _mat[_i, _j]
            _lv = _log_mat[_i, _j]
            _txt = "1×" if _i == _j else f"{_v:.2f}×"
            ax.text(_j, _i, _txt, ha="center", va="center", fontsize=7,
                    color="white" if abs(_lv) > _abs_max * 0.6 else "black")

    fig.suptitle("E1 — Pairwise speedups between quantizations (tok/s)",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("No data for speedup table")


In [ ]:
# Bar plot of speedups relative to Q4_K_M (common reference in the literature)
_REF_QUANT = "Q4_K_M"
_q_col_ref  = "quantization" if "quantization" in ex.columns else "model_short"

if not ex.empty and "tokens_per_s_mean" in ex.columns and _REF_QUANT in ex[_q_col_ref].values:
    _df_ref = ex.dropna(subset=["tokens_per_s_mean"]).copy()

    # Median tok/s per (model_label, quantization)
    _speed_ref = (
        _df_ref.groupby(["model_label", _q_col_ref])["tokens_per_s_mean"]
        .median().reset_index()
    )
    _ref_vals = (
        _speed_ref[_speed_ref[_q_col_ref] == _REF_QUANT]
        .set_index("model_label")["tokens_per_s_mean"]
    )
    _speed_ref = _speed_ref.copy()
    _speed_ref["ref_speed"]     = _speed_ref["model_label"].map(_ref_vals)
    _speed_ref = _speed_ref.dropna(subset=["ref_speed"])
    _speed_ref["speedup_vs_ref"] = _speed_ref["tokens_per_s_mean"] / _speed_ref["ref_speed"]

    _quant_order_plot = [q for q in QUANT_ORDER if q in _speed_ref[_q_col_ref].values]
    _models_u = sorted(_speed_ref["model_label"].unique())

    fig, ax = plt.subplots(figsize=(14, 6))
    _x_pos = np.arange(len(_quant_order_plot))
    _n_m   = len(_models_u)
    _width = 0.7 / max(_n_m, 1)
    _BAR_COLORS = ["#377eb8","#e41a1c","#4daf4a","#ff7f00","#984ea3","#a65628","#f781bf","#999999"]

    for _j, _model in enumerate(_models_u):
        _sub = _speed_ref[_speed_ref["model_label"] == _model]
        _vals = [
            _sub[_sub[_q_col_ref] == q]["speedup_vs_ref"].values[0]
            if len(_sub[_sub[_q_col_ref] == q]) > 0 else float("nan")
            for q in _quant_order_plot
        ]
        _offset = (_j - _n_m / 2 + 0.5) * _width
        ax.bar(_x_pos + _offset, _vals, _width * 0.85,
               color=_BAR_COLORS[_j % len(_BAR_COLORS)], alpha=0.85, label=_model)

    ax.axhline(1.0, ls="--", color="black", linewidth=1.2,
               label=f"Reference ({_REF_QUANT} = 1×)")
    ax.set_xticks(_x_pos)
    ax.set_xticklabels(_quant_order_plot, rotation=30, ha="right")
    ax.set_ylabel(f"Speedup relative to {_REF_QUANT}")
    ax.set_xlabel("Quantization")
    fig.suptitle(f"E1 — Speedup relative to {_REF_QUANT} by model (tok/s)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(True, alpha=0.3, axis="y")

    fig.tight_layout()
    plt.show()

    _pivot = _speed_ref.pivot_table(
        index="model_label", columns=_q_col_ref, values="speedup_vs_ref"
    )[_quant_order_plot].round(2)
    print(f"\nSpeedup relative to {_REF_QUANT} (tok/s):")
    display(_pivot)
else:
    print(f"No {_REF_QUANT} data to normalize speedups")


## Conclusions and discussion

### Hypothesis confirmation/refutation
> TODO

### Key findings
> TODO

### Limitations
> TODO
